# Workshop: Lakeflow Declarative Pipeline

**Scenario:** Build a complete RetailHub medallion pipeline and then explain why each layer exists. First you will wire the pipeline in the Databricks UI, then you will write the Lakeflow SQL declarations and verify the result.

This lab mirrors the full production flow:
1. Build and run the pipeline in the UI
2. Configure pipeline variables and source assets
3. Declare Bronze, Silver, and Gold layers in SQL
4. Inspect the Event Log and compare STREAMING TABLE vs MATERIALIZED VIEW

## Learning Focus

- Build a Lakeflow pipeline end-to-end in the UI
- Use `STREAMING TABLE` for Bronze ingestion
- Apply `EXPECT ... ON VIOLATION DROP ROW` for Silver quality gates
- Use `MATERIALIZED VIEW` for Gold aggregations
- Read the Event Log to verify pipeline behavior

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Lakeflow pipeline target schema (matches Step 2 config)
user_name = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
user_schema = f"{user_name}_lakeflow"
print(f"Pipeline target schema: {user_schema}")

## Section 1: Workshop — Building the Pipeline

In this hands-on workshop, we create a complete Lakeflow pipeline from SQL files — uploading source code, configuring the pipeline in the Databricks UI, running it, and validating the results.



### SQL Files Overview

The pipeline source code is organized by medallion layer — each SQL file declares one table or view.

![../../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp](../../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp)

### Step 1: Upload SQL Files

**Option A: Via UI** — Workspace → Users → create `lakeflow_pipeline` folder → upload SQL files

**Option B: Via Git Folders** — Git Folders → Add Git Folder → clone training repository

### Step 2: Create Pipeline

1. **Workflows** → **Jobs & Pipelines** → **Create ETL Pipeline**
2. **Catalog:** `retailhub_<your_name>`
3. **Pipeline Name:** `lakeflow_pipeline_<your_name>`
4. **Target Schema:** `<your_name>_lakeflow`
5. **Source Code:** Add existing assets → choose pipeline root folder and source code folder

<img src="../../../assets/images/fab4fef8e72d4d5786ba818e6c2f73c5.png" width="800">

<img src="../../../assets/images/a4e21cb35d3b45f2ba184877139cdc48.png" width="800">

<img src="../../../assets/images/4e75e96544f142508c3bd8b0b4dbf446.png" width="800">

![../../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp](../../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp)

### Step 3: Configure Variables

**Configuration** → **Add configuration**:

| Key | Value |
|-----|-------|
| `customer_path` | `/Volumes/<your catalog>/default/datasets/customers` |
| `order_path` | `/Volumes/<your catalog>/default/datasets/orders` |
| `product_path` | `/Volumes/<your catalog>/default/datasets/products/products.parquet/` |

Open settings and go to Pipeline Configuration 

<img src="../../../assets/images/b2fa841a52004939ac2679f9edef8dc3.png" width="800">

Add configuration : 

<img src="../../../assets/images/dc31d5dec7444c1d959b8e1f220c10ce.png" width="800">

<img src="../../../assets/images/6f16ad4f7fd941ebbb4fb286dbb8fbfd.png" width="800">

You should also see a DAG diagram built based on Spark Declarative Pipelines definition

![../../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp](../../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp)

### Step 4: Run the Pipeline

Start the pipeline and test incremental processing by adding new data files.

![../../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp](../../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp)

1. Add new file to folder orders/stream/ from /Volumes/.../default/datasets/demo/ingestion/orders/stream/
2. Run pipeline again
3. Check Event Log - should process only new files

### Step 5: Verify Results

In [ ]:
# Check fact_sales with joins to dimensions
display(spark.sql(f"""
    SELECT 
        f.order_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        p.product_name,
        d.date,
        f.quantity,
        f.net_amount
    FROM {CATALOG}.{user_schema}.fact_sales f
    LEFT JOIN {CATALOG}.{user_schema}.dim_customer c ON f.customer_key = c.customer_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_product p ON f.product_key = p.product_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_date d ON f.order_date_key = d.date_key
    LIMIT 10
"""))

In [ ]:
# Find customers with change history
display(spark.sql(f"""
    SELECT 
        customer_id, first_name, city,
        __START_AT, __END_AT,
        CASE WHEN __END_AT IS NULL THEN 'Current' ELSE 'Historical' END AS status
    FROM {CATALOG}.{user_schema}.silver_customers
    WHERE customer_id IN (
        SELECT customer_id 
        FROM {CATALOG}.{user_schema}.silver_customers 
        GROUP BY customer_id HAVING COUNT(*) > 1
    )
    ORDER BY customer_id, __START_AT
"""))

### Monitoring and Troubleshooting

Common issues encountered when running Lakeflow pipelines and how to resolve them.

| Issue | Cause | Solution |
|---------|-----------|-------------|
| Pipeline hangs | Cluster too small | Increase min workers |
| Missing data | Constraint DROP ROW | Check Data Quality tab |
| Schema mismatch | Schema change | Full refresh |

## Section 2: Practice — Lakeflow SQL Declarations

Write and verify Lakeflow SQL syntax for each medallion layer.

## Task 1: Write Bronze Declaration

Complete the SQL below to create a Bronze streaming table from JSON files.

**What you need to do:** Fill in the blanks:
1. `CREATE OR REFRESH ... TABLE` — keyword to fill: `STREAMING`
2. `STREAM ...(...)` — function to fill: `read_files`

**This SQL would go in a pipeline SQL file.** Here we practice the syntax.

**Guidance — Task 01**

The goal is to declare a **Bronze streaming table** that ingests raw JSON files using Lakeflow's `read_files` function.

**STREAMING TABLE vs regular table**
A `STREAMING TABLE` processes data incrementally — it tracks what has been read and only picks up new files on subsequent pipeline runs. This is the standard pattern for Bronze ingestion. The `STREAM` keyword before `read_files(...)` enables incremental file processing: Lakeflow creates a checkpoint and on re-run processes only new files, not the entire directory.

**Why `read_files` instead of `spark.read.format(...)`?**
`read_files()` is the Lakeflow-native function for file ingestion. It integrates with Auto Loader, handles schema evolution, and supports all common formats (`json`, `csv`, `parquet`). Unlike `spark.read`, it supports both batch (`read_files`) and streaming (`STREAM read_files`) modes within a Lakeflow pipeline.

In [ ]:
# Practice: write the Bronze SQL declaration syntax
# (This won't execute outside of a Lakeflow pipeline, but verify the syntax)

# TODO: Complete the Bronze declaration
# The keyword before TABLE is: STREAMING
# The function to read from a file path in Lakeflow is: read_files(path, format => ...)
bronze_sql = f"""
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * 
FROM STREAM read_files(
    -- TODO: your source path and format here
);
"""

print("Bronze SQL declaration:")
print(bronze_sql)

In [ ]:
# -- Validation --
assert "STREAMING" in bronze_sql.upper(), "Should use STREAMING TABLE"
assert "read_files" in bronze_sql.lower(), "Should use read_files()"
print("Task 1 OK: Bronze declaration syntax correct")

## Task 2: Write Silver Declaration with Expectations

Complete the Silver layer with data quality constraints.

**What you need to do:** Complete `ON VIOLATION ...` — use keyword: `DROP ROW` for both constraints.

**Guidance — Task 02**

The goal is to add **data quality expectations** to the Silver layer — the first line of defense against bad data.

**How expectations work**
Expectations are `CONSTRAINT` declarations in the `CREATE` statement. Each constraint has a name, a boolean expression, and a violation action.

| Action | Behavior | When to use |
|--------|----------|-------------|
| `ON VIOLATION DROP ROW` | Row removed silently; violation logged | Business rule violations (null IDs, negative quantities) |
| `ON VIOLATION FAIL UPDATE` | Entire pipeline run fails | Critical integrity constraints — no data is better than bad data |
| *(no action)* | Row kept; violation count logged only | Monitoring without blocking |

**Exam Tip:** `DROP ROW` is the most commonly tested action. Know that it silently discards the row and logs the violation count in the Event Log — it does NOT fail the pipeline.

In [ ]:
# TODO: Complete Silver SQL with data quality expectations
# ON VIOLATION options:
#   DROP ROW  — silently drop rows that violate the constraint
#   FAIL      — fail the entire pipeline run if any row violates
# Use DROP ROW for recoverable data issues, FAIL for critical integrity constraints

silver_sql = """
CREATE OR REFRESH STREAMING TABLE silver_orders (
    CONSTRAINT valid_id EXPECT (order_id IS NOT NULL) ON VIOLATION ...,
    CONSTRAINT positive_amount EXPECT (total_price > 0) ON VIOLATION ...
)
AS SELECT 
    order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(total_price AS DOUBLE) AS total_price,
    CAST(order_date AS DATE) AS order_date,
    payment_method,
    store_id,
    current_timestamp() AS processed_at
FROM STREAM(bronze_orders);
"""

print("Silver SQL declaration:")
print(silver_sql)

In [ ]:
# -- Validation --
assert "CONSTRAINT" in silver_sql.upper(), "Should have CONSTRAINT declarations"
assert "DROP ROW" in silver_sql.upper(), "Should use ON VIOLATION DROP ROW"
assert "bronze" in silver_sql.lower(), "Should reference bronze_orders"
print("Task 2 OK: Silver declaration with expectations correct")

## Task 3: Write Gold Declaration

Create a Materialized View for daily revenue summary.

**What you need to do:** Complete `CREATE OR REFRESH ...` — use: `MATERIALIZED VIEW`

**Guidance — Task 03**

The goal is to write a **Gold Materialized View** — the final aggregated layer optimized for analytics.

**MATERIALIZED VIEW vs STREAMING TABLE**
A `MATERIALIZED VIEW` is recomputed from scratch on each pipeline run — Lakeflow re-reads the entire source and replaces the result. This is correct for aggregations (`SUM`, `COUNT`, `GROUP BY`) where appending would produce wrong totals.

A `STREAMING TABLE` at Gold (like `fact_sales`) processes records incrementally — each batch from Silver adds new rows. This works for fact tables where records are immutable.

**Rule of thumb:** `MATERIALIZED VIEW` for dimension tables and aggregated summaries. `STREAMING TABLE` for fact tables and append-only datasets.

In [ ]:
# TODO: Complete the Gold declaration
# Gold reads from silver_orders which is a BATCH aggregation (not streaming)
# Use LIVE TABLE (not STREAMING TABLE) for batch/aggregate tables

gold_sql = """
-- TODO: Write the full CREATE OR REFRESH ... TABLE gold_daily_revenue declaration
-- Include the AS SELECT aggregation below
AS SELECT 
    order_date,
    SUM(total_price) AS total_revenue,
    COUNT(*) AS total_orders,
    AVG(total_price) AS avg_order_value
FROM silver_orders
GROUP BY order_date
ORDER BY order_date;
"""

print("Gold SQL declaration:")
print(gold_sql)

In [ ]:
# -- Validation --
assert "MATERIALIZED VIEW" in gold_sql.upper(), "Gold should use MATERIALIZED VIEW"
assert "silver" in gold_sql.lower(), "Should reference silver_orders"
print("Task 3 OK: Gold Materialized View declaration correct")

## Task 4: Compare STREAMING TABLE vs MATERIALIZED VIEW

Fill in the comparison table (markdown exercise).

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Processing mode | Incremental (append-only) | Full refresh or incremental |
| Best for | Append-only sources (files, CDC) | Aggregations, joins, computed columns |
| Read from source | `STREAM(table_name)` | `table_name` |
| Supports expectations | Yes | Yes |

**What you need to do:** Fill in the four blank cells above (Processing mode and Best for).

**Guidance — Task 04**

The goal is to solidify your understanding of the **difference between STREAMING TABLE and MATERIALIZED VIEW** — a core concept tested in the DEA exam.

**Processing mode**
- `STREAMING TABLE` → **Incremental**: processes only new records since last run
- `MATERIALIZED VIEW` → **Full recomputation**: re-reads entire source on each refresh

**Filling in the table:**
- Processing mode: `Incremental` | `Full recomputation`
- Best for: `Bronze / Silver ingestion` | `Gold aggregations / dimension tables`

**Exam Tip:** If a query uses `FROM STREAM(table)`, the target **must** be a `STREAMING TABLE`. `MATERIALIZED VIEW` cannot reference `STREAM(...)`.

## Task 5: Verify Pipeline Results (after running the pipeline)

After creating and running the pipeline via the UI, query the results here.

**What you need to do:** Uncomment the queries and run them to verify tables exist with data.

**Guidance — Task 05**

The goal is to **verify the pipeline ran correctly** by querying the output tables in each medallion layer.

**Where pipeline tables live**
All tables declared in the pipeline are created under the Target Schema you configured (`catalog.schema.table_name`). They are standard Delta tables — query them with `spark.sql()`.

**What to check:**
- Bronze: raw row count (should match source files)
- Silver: row count should be <= Bronze (some rows dropped by expectations)
- Gold `fact_sales`: verify joins resolved (`customer_key != -1` ratio)
- `silver_customers`: look for `__END_AT IS NOT NULL` records — those are historical versions from SCD2 changes

In [ ]:
# TODO: After running the pipeline, uncomment and verify:

# display(spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.bronze.bronze_orders"))
# display(spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.silver.silver_orders"))
# display(spark.sql(f"SELECT * FROM {CATALOG}.gold.gold_daily_revenue ORDER BY order_date"))

## Task 6: Check Pipeline Event Log

Query the pipeline event log to see data quality metrics.

**What you need to do:** Uncomment the query and check how many rows were dropped by the Silver expectations.

**Guidance — Task 06**

The goal is to query the **pipeline Event Log** — a built-in audit trail that records every pipeline action.

**What the Event Log captures**
Every pipeline run generates events: schema inference, data quality checks, row counts, errors, and lineage. The `event_log()` table function exposes these as a queryable Delta table.

**JSON path navigation with `:`**
Databricks SQL uses the `:` operator for nested JSON: `details:flow_progress:data_quality:expectations` extracts the expectations sub-object. This is equivalent to `details['flow_progress']['data_quality']['expectations']` in Python.

**What to look for:**
- `num_rows_dropped` — rows dropped by `ON VIOLATION DROP ROW`
- `pass_ratio` — percentage of rows passing the constraint
- High drop rates indicate upstream data quality issues

In [ ]:
# TODO: After running the pipeline, uncomment to check event log:

# display(spark.sql("""
#     SELECT timestamp, details:flow_progress:data_quality:expectations
#     FROM event_log(TABLE({CATALOG}.silver.silver_orders))
#     WHERE details:flow_progress:data_quality IS NOT NULL
# """))

## Lab Complete!

You have:
- Written Bronze STREAMING TABLE declarations
- Written Silver declarations with data quality expectations
- Written Gold MATERIALIZED VIEW declarations
- Understood ST vs MV differences
- (If pipeline ran) Verified results and checked data quality metrics

> **Exam Tip:** In Spark Declarative Pipelines, tables within the same pipeline reference each other directly by name — no prefix needed. Use `STREAM(table_name)` for streaming reads and just `table_name` for batch reads.

> **Next:** LAB 08 - Lakeflow Jobs & Orchestration

## Cleanup (Optional)

In [ ]:
# Pipeline cleanup is done via Lakeflow UI (delete the pipeline)
print("LAB 07 complete. Delete the pipeline from Lakeflow UI when done.")

← [07 — Lakeflow Pipelines](../demo/07_lakeflow_pipelines.ipynb) | **[ README](../../../README.md)** | [08 — Job Orchestration →](../demo/08_job_orchestration.ipynb)